In [10]:
import os
from pprint import pprint
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import psycopg
import pandas as pd
from typing import Any
import json
from psycopg import sql
load_dotenv()
MODELO='gemini-3.1-flash-lite'
generador_consultas=ChatGoogleGenerativeAI(model=MODELO,temperature=0)
model=ChatGoogleGenerativeAI(model=MODELO)
parser=StrOutputParser()

In [ ]:
consulta_messages=[('system',"""
        Lenguaje: sql pgAdmin
        Tabla: v_viviendas
        Columnas: [nombre, localidad, cp, precio, tipo, superficie_util, conservacion, superficie_solar, planta, habitaciones, banhos, referencia, antiguedad, superficie_construida, adaptado_a_personas_con_movilidad_reducida, balcon, se_aceptan_mascotas, calle_alumbrada, calle_asfaltada, lavadero, ascensor, chimenea, cocina_equipada, exterior, armarios_empotrados, garaje, sistema_de_seguridad, piscina, comedor, luz, carpinteria_exterior, orientacion, calefaccion, amueblado, urbanizado, trastero, puerta_blindada, vidrios_dobles, telefono, interior, soleado, agua, portero_automatico, aire_acondicionado, tipo_suelo, jardin, carpinteria_interior, terraza, gastos_de_comunidad, gas, precio_superficie, consumo, emisiones, descripcion]
        Petición: {peticion}
        Tarea: Select de sql que dé lo que la petición quiere. NADA MÁS QUE ESO. SIN ADORNOS
        Salida: SELECT ... where ... is not null ...
        Importante: Eliminar los nulos en las columnas requeridas. Que la consulta conteste a la peticion con las mínimas filas necesarias pero incluyendo la descripción
    """)]
def sacar_consulta(peticion:str)->str:
    global consulta_messages
    template="""
        Lenguaje: sql pgAdmin
        Tabla: v_viviendas
        Columnas: [nombre, localidad, cp, precio, tipo, superficie_util, conservacion, superficie_solar, planta, habitaciones, banhos, referencia, antiguedad, superficie_construida, adaptado_a_personas_con_movilidad_reducida, balcon, se_aceptan_mascotas, calle_alumbrada, calle_asfaltada, lavadero, ascensor, chimenea, cocina_equipada, exterior, armarios_empotrados, garaje, sistema_de_seguridad, piscina, comedor, luz, carpinteria_exterior, orientacion, calefaccion, amueblado, urbanizado, trastero, puerta_blindada, vidrios_dobles, telefono, interior, soleado, agua, portero_automatico, aire_acondicionado, tipo_suelo, jardin, carpinteria_interior, terraza, gastos_de_comunidad, gas, precio_superficie, consumo, emisiones, descripcion]
        Petición: {peticion}
        Tarea: Select de sql que dé lo que la petición quiere. NADA MÁS QUE ESO. SIN ADORNOS
        Salida: SELECT ... where ... is not null ...
        Importante: Eliminar los nulos en las columnas requeridas. Que la consulta conteste a la peticion con las mínimas filas necesarias pero incluyendo la descripción
    """
    prompt=PromptTemplate(input_variables=['peticion'],template=template)
    return ((prompt | generador_consultas | parser).invoke(input={"peticion":peticion})).replace(';','').replace('sql ','')

In [16]:
messages=[('system','ERES ANALISTA DEL MERCADO INMOBILIARIO DE MADRID. NO TE PRESENTES. CÍÑETE SOBRETODO A LOS DATOS QUE SE TE PROPORCIONA. HABLA DE LOS DATOS COMO SI NO FUERA EL USUARIO QUIEN TE LOS DIÓ. USA LAS DESCRIPCIONES SI TE ES ÚTIL. TODOS LOS PISOS ESTÁN EN VENTA.')]
while True:
    request=input('User (introduce esc para salir): ')
    if request.lower() in ['esc']:
        break
    q=sacar_consulta(request)
    query=sql.SQL("select row_to_json(t) from ({subquery}) t;").format(
        subquery=sql.SQL(q)
    )
    async with await psycopg.AsyncConnection.connect(str(os.environ.get('URL'))) as connection:
        async with connection.cursor() as cursor:
            await cursor.execute(query)
            filas=await cursor.fetchall()
            resultado=[fila[0] for fila in filas]
    messages.extend([('ai',f'ANUNCIOS DE REFERENCIA: {json.dumps(resultado).replace('{','{{').replace('}','}}')}'),('human','{request}')])
    prompt_chat=ChatPromptTemplate.from_messages(messages=messages)
    cadena=prompt_chat | model | parser
    respuesta=cadena.astream(input={"request":request})
    completo:str=''
    async for chunk in respuesta:
        completo+=chunk
        print(chunk,end='',flush=True)
    messages.append(('ai',completo))

Basado en el inventario actual de inmuebles en venta, las opciones más económicas se concentran en el segmento de activos de rehabilitación o estructuras móviles, situándose principalmente fuera de la capital:

1.  **Corralejos:** Se encuentra la opción con el precio más bajo, una casa móvil de bambú por **11.500 €**, equipada con dos habitaciones y mobiliario básico.
2.  **Paredes de Buitrago:** En la Sierra Norte de Madrid, existen tres oportunidades destacadas para rehabilitación:
    *   Un pajar para reformar de 37 m² por **16.800 €**.
    *   Un pajar para reformar de 42 m² por **19.000 €**.
    *   Una casa de 45 m² a reformar por **23.000 €**.
3.  **Daganzo de Arriba:** Se ofrece una parcela en un camping privado con casa de madera por **25.000 €**. Cabe señalar que este inmueble está sujeto a normativa de uso recreativo, con estancias limitadas y sin posibilidad de residencia permanente.

En contraste, los inmuebles residenciales convencionales situados dentro de Madrid capita